In [4]:
import pandas as pd
import json
import os

def generate_multimodal_jsonl(csv_path, output_jsonl):
    """
    Converts a CSV with relative image paths into a Qwen2.5-VL compatible JSONL.
    
    Args:
        csv_path (str): Path to your source CSV.
        output_jsonl (str): Path to save the generated JSONL.
        image_root_dir (str): The base directory where your images are stored.
    """
    # Load your 33k+ rows
    df = pd.read_csv(csv_path)
    
    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            # Convert binary Label to "Yes"/"No" string
            label_str = "Yes" if str(row['Label']) == '1' else "No"
            
            # Construct absolute paths with file:// protocol for the processor
            img1_path = os.path.abspath(os.path.join(str(row['image1'])))
            img2_path = os.path.abspath(os.path.join(str(row['image2'])))
            
            # Standard Qwen2.5-VL ChatML structure
            payload = {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": f"{img1_path}"},
                            {"type": "image", "image": f"{img2_path}"},
                            {
                                "type": "text", 
                                "text": f"Product 1 Title: {row['title1']}\nProduct 2 Title: {row['title2']}\nAre these the exact same product? Answer Yes or No."
                            }
                        ]
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": label_str}
                        ]
                    }
                ]
            }
            
            f.write(json.dumps(payload, ensure_ascii=False) + "\n")

    print(f"✅ Successfully generated {len(df)} rows to {output_jsonl}")

# --- Execution ---

# Replace these with your actual paths
CSV_FILE = "train_pairs.csv"
OUTPUT_FILE = "train_pairs_vl.jsonl"
#IMAGE_FOLDER = "C:/Users/YourName/Pictures/Products" # Your base folder
generate_multimodal_jsonl( "train_pairs.csv", "train_pairs_vl.jsonl")
generate_multimodal_jsonl( "test_pairs.csv", "test_pairs_vl.jsonl")


✅ Successfully generated 134001 rows to train_pairs_vl.jsonl
✅ Successfully generated 33501 rows to test_pairs_vl.jsonl
